# Lab Tùy chọn: Gradient Descent cho Hồi quy Tuyến tính

<figure>
    <center> <img src="./images/C1_W1_L4_S1_Lecture_GD.png"  style="width:800px;height:200px;" ></center>
</figure>

## Mục tiêu
Trong lab này, bạn sẽ:
- tự động hóa quá trình tối ưu hóa $w$ và $b$ bằng gradient descent.

## Công cụ
Trong lab này, chúng ta sẽ sử dụng: 
- NumPy, một thư viện phổ biến cho tính toán khoa học
- Matplotlib, một thư viện phổ biến để vẽ đồ thị
- các hàm vẽ đồ thị trong file lab_utils.py ở thư mục hiện tại

In [ ]:
import math, copy
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('./deeplearning.mplstyle')
from lab_utils_uni import plt_house_x, plt_contour_wgrad, plt_divergence, plt_gradients

<a name="toc_40291_2"></a>
# Phát biểu bài toán

Hãy sử dụng hai điểm dữ liệu giống như trước - một căn nhà 1000 feet vuông được bán với giá \\$300,000 và một căn nhà 2000 feet vuông được bán với giá \\$500,000.

| Diện tích (1000 sqft)     | Giá (1000 đô la) |
| ----------------| ------------------------ |
| 1               | 300                      |
| 2               | 500                      |


In [ ]:
# Nạp tập dữ liệu của chúng ta
x_train = np.array([1.0, 2.0])   #đặc trưng
y_train = np.array([300.0, 500.0])   #giá trị mục tiêu

<a name="toc_40291_2.0.1"></a>
### Compute_Cost
Hàm này đã được xây dựng trong lab trước. Chúng ta sẽ cần dùng lại nó ở đây.

In [ ]:
#Hàm để tính cost
def compute_cost(x, y, w, b):
   
    m = x.shape[0] 
    cost = 0
    
    for i in range(m):
        f_wb = w * x[i] + b
        cost = cost + (f_wb - y[i])**2
    total_cost = 1 / (2 * m) * cost

    return total_cost

<a name="toc_40291_2.1"></a>
## Tóm tắt về Gradient Descent
Cho đến nay trong khóa học này, bạn đã xây dựng một mô hình tuyến tính dự đoán $f_{w,b}(x^{(i)})$:
$$f_{w,b}(x^{(i)}) = wx^{(i)} + b \tag{1}$$
Trong hồi quy tuyến tính, bạn sử dụng dữ liệu huấn luyện đầu vào để khớp các tham số $w$,$b$ bằng cách tối thiểu hóa một thước đo lỗi giữa các dự đoán $f_{w,b}(x^{(i)})$ của chúng ta và dữ liệu thực tế $y^{(i)}$. Thước đo này được gọi là $cost$ (chi phí), $J(w,b)$. Trong quá trình huấn luyện, bạn đo cost trên tất cả các mẫu huấn luyện $x^{(i)},y^{(i)}$
$$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2\tag{2}$$ 


Trong bài giảng, *gradient descent* được mô tả là:

$$\begin{align*} \text{lặp lại}&\text{ cho đến khi hội tụ:} \; \lbrace \newline
\;  w &= w -  \alpha \frac{\partial J(w,b)}{\partial w} \tag{3}  \; \newline 
 b &= b -  \alpha \frac{\partial J(w,b)}{\partial b}  \newline \rbrace
\end{align*}$$
trong đó, các tham số $w$, $b$ được cập nhật đồng thời.  
Gradient được định nghĩa là:
$$
\begin{align}
\frac{\partial J(w,b)}{\partial w}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})x^{(i)} \tag{4}\\
  \frac{\partial J(w,b)}{\partial b}  &= \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)}) \tag{5}\\
\end{align}
$$

Ở đây *đồng thời* nghĩa là bạn tính các đạo hàm riêng cho tất cả các tham số trước khi cập nhật bất kỳ tham số nào.

<a name="toc_40291_2.2"></a>
## Triển khai Gradient Descent
Bạn sẽ triển khai thuật toán gradient descent cho một đặc trưng. Bạn sẽ cần ba hàm. 
- `compute_gradient` triển khai phương trình (4) và (5) ở trên
- `compute_cost` triển khai phương trình (2) ở trên (mã từ lab trước)
- `gradient_descent`, sử dụng compute_gradient và compute_cost

Quy ước:
- Việc đặt tên biến python chứa đạo hàm riêng tuân theo mẫu này, $\frac{\partial J(w,b)}{\partial b}$  sẽ là `dj_db`.
- w.r.t là viết tắt của With Respect To (theo/đối với), như trong đạo hàm riêng của $J(wb)$ theo $b$.


<a name="toc_40291_2.3"></a>
### compute_gradient
<a name='ex-01'></a>
`compute_gradient`  triển khai (4) và (5) ở trên và trả về $\frac{\partial J(w,b)}{\partial w}$,$\frac{\partial J(w,b)}{\partial b}$. Các comment nhúng trong code mô tả các phép toán.

In [ ]:
def compute_gradient(x, y, w, b): 
    """
    Tính gradient cho hồi quy tuyến tính 
    Args:
      x (ndarray (m,)): Dữ liệu, m ví dụ 
      y (ndarray (m,)): giá trị mục tiêu
      w,b (scalar)    : tham số mô hình  
    Returns
      dj_dw (scalar): Gradient của cost theo tham số w
      dj_db (scalar): Gradient của cost theo tham số b     
     """
    
    # Số lượng ví dụ huấn luyện
    m = x.shape[0]    
    dj_dw = 0
    dj_db = 0
    
    for i in range(m):  
        f_wb = w * x[i] + b 
        dj_dw_i = (f_wb - y[i]) * x[i] 
        dj_db_i = f_wb - y[i] 
        dj_db += dj_db_i
        dj_dw += dj_dw_i 
    dj_dw = dj_dw / m 
    dj_db = dj_db / m 
        
    return dj_dw, dj_db

<br/>

<img align="left" src="./images/C1_W1_Lab03_lecture_slopes.PNG"   style="width:340px;" > Bài giảng đã mô tả cách gradient descent sử dụng đạo hàm riêng của cost theo một tham số tại một điểm để cập nhật tham số đó.   
Hãy sử dụng hàm `compute_gradient` của chúng ta để tìm và vẽ một số đạo hàm riêng của hàm cost theo một trong các tham số, $w_0$.


In [ ]:
plt_gradients(x_train,y_train, compute_cost, compute_gradient)
plt.show()

Ở trên, đồ thị bên trái cho thấy $\frac{\partial J(w,b)}{\partial w}$ hay độ dốc của đường cong cost theo $w$ tại ba điểm. Ở phía bên phải của đồ thị, đạo hàm là dương, trong khi ở bên trái nó là âm. Do 'hình dạng cái tô', các đạo hàm sẽ luôn dẫn gradient descent về phía đáy nơi gradient bằng 0.
 
Đồ thị bên trái cố định $b=100$. Gradient descent sẽ sử dụng cả $\frac{\partial J(w,b)}{\partial w}$ và $\frac{\partial J(w,b)}{\partial b}$ để cập nhật các tham số. 'Đồ thị quiver' (mũi tên) bên phải cung cấp một cách để xem gradient của cả hai tham số. Kích thước mũi tên phản ánh độ lớn của gradient tại điểm đó. Hướng và độ dốc của mũi tên phản ánh tỷ lệ giữa $\frac{\partial J(w,b)}{\partial w}$ và $\frac{\partial J(w,b)}{\partial b}$ tại điểm đó.
Lưu ý rằng gradient chỉ theo hướng *ra xa* điểm cực tiểu. Xem lại phương trình (3) ở trên. Gradient đã được nhân với hệ số tỷ lệ sẽ được *trừ* khỏi giá trị hiện tại của $w$ hoặc $b$. Điều này di chuyển tham số theo hướng làm giảm cost.

<a name="toc_40291_2.5"></a>
###  Gradient Descent
Bây giờ khi gradient đã có thể được tính toán, gradient descent, được mô tả trong phương trình (3) ở trên có thể được triển khai bên dưới trong `gradient_descent`. Chi tiết triển khai được mô tả trong các comment. Bên dưới, bạn sẽ sử dụng hàm này để tìm các giá trị tối ưu của $w$ và $b$ trên dữ liệu huấn luyện.

In [ ]:
def gradient_descent(x, y, w_in, b_in, alpha, num_iters, cost_function, gradient_function): 
    """
    Thực hiện gradient descent để khớp w,b. Cập nhật w,b bằng cách thực hiện 
    num_iters bước gradient với tốc độ học alpha
    
    Args:
      x (ndarray (m,))  : Dữ liệu, m ví dụ 
      y (ndarray (m,))  : giá trị mục tiêu
      w_in,b_in (scalar): giá trị khởi tạo của các tham số mô hình  
      alpha (float):     Tốc độ học (learning rate)
      num_iters (int):   số lần lặp để chạy gradient descent
      cost_function:     hàm được gọi để tính cost
      gradient_function: hàm được gọi để tính gradient
      
    Returns:
      w (scalar): Giá trị đã cập nhật của tham số sau khi chạy gradient descent
      b (scalar): Giá trị đã cập nhật của tham số sau khi chạy gradient descent
      J_history (List): Lịch sử các giá trị cost
      p_history (list): Lịch sử các tham số [w,b] 
      """
    
    # Một mảng để lưu cost J và các giá trị w tại mỗi lần lặp, chủ yếu để vẽ đồ thị sau này
    J_history = []
    p_history = []
    b = b_in
    w = w_in
    
    for i in range(num_iters):
        # Tính gradient và cập nhật các tham số bằng gradient_function
        dj_dw, dj_db = gradient_function(x, y, w , b)     

        # Cập nhật Tham số sử dụng phương trình (3) ở trên
        b = b - alpha * dj_db                            
        w = w - alpha * dj_dw                            

        # Lưu cost J tại mỗi lần lặp
        if i<100000:      # ngăn cạn kiệt tài nguyên 
            J_history.append( cost_function(x, y, w , b))
            p_history.append([w,b])
        # In cost mỗi 10 khoảng lần lặp, hoặc nhiều lần lặp hơn nếu < 10
        if i% math.ceil(num_iters/10) == 0:
            print(f"Lần lặp {i:4}: Cost {J_history[-1]:0.2e} ",
                  f"dj_dw: {dj_dw: 0.3e}, dj_db: {dj_db: 0.3e}  ",
                  f"w: {w: 0.3e}, b:{b: 0.5e}")
 
    return w, b, J_history, p_history #trả về w và lịch sử J,w để vẽ đồ thị

In [ ]:
# khởi tạo tham số
w_init = 0
b_init = 0
# một số thiết lập gradient descent
iterations = 10000
tmp_alpha = 1.0e-2
# chạy gradient descent
w_final, b_final, J_hist, p_hist = gradient_descent(x_train ,y_train, w_init, b_init, tmp_alpha, 
                                                    iterations, compute_cost, compute_gradient)
print(f"(w,b) tìm được bởi gradient descent: ({w_final:8.4f},{b_final:8.4f})")

<img align="left" src="./images/C1_W1_Lab03_lecture_learningrate.PNG"  style="width:340px; padding: 15px; " > 
Hãy dành chút thời gian và chú ý một số đặc điểm của quá trình gradient descent được in ở trên.  

- Cost bắt đầu lớn và giảm nhanh chóng như được mô tả trong slide bài giảng.
- Các đạo hàm riêng, `dj_dw`, và `dj_db` cũng nhỏ dần, nhanh ở giai đoạn đầu và sau đó chậm hơn. Như được thể hiện trong sơ đồ từ bài giảng, khi quá trình tiến gần đến 'đáy của cái tô', tiến trình chậm hơn do giá trị đạo hàm nhỏ hơn tại điểm đó.
- tiến trình chậm lại mặc dù tốc độ học, alpha, vẫn không đổi

### Cost theo số lần lặp của gradient descent 
Một đồ thị cost theo số lần lặp là một thước đo hữu ích cho tiến trình của gradient descent. Cost nên luôn giảm trong các lần chạy thành công. Sự thay đổi của cost ở giai đoạn đầu rất nhanh, nên việc vẽ đồ thị phần giảm ban đầu trên một tỷ lệ khác với phần giảm cuối cùng là hữu ích. Trong các đồ thị bên dưới, hãy chú ý tỷ lệ của cost trên các trục và bước lặp.

In [ ]:
# vẽ đồ thị cost theo số lần lặp  
fig, (ax1, ax2) = plt.subplots(1, 2, constrained_layout=True, figsize=(12,4))
ax1.plot(J_hist[:100])
ax2.plot(1000 + np.arange(len(J_hist[1000:])), J_hist[1000:])
ax1.set_title("Cost theo số lần lặp (đầu)");  ax2.set_title("Cost theo số lần lặp (cuối)")
ax1.set_ylabel('Cost')            ;  ax2.set_ylabel('Cost') 
ax1.set_xlabel('bước lặp')  ;  ax2.set_xlabel('bước lặp') 
plt.show()

### Dự đoán
Bây giờ khi bạn đã tìm ra các giá trị tối ưu cho các tham số $w$ và $b$, bạn có thể sử dụng mô hình để dự đoán giá trị nhà dựa trên các tham số đã học được. Như mong đợi, các giá trị dự đoán gần như giống với các giá trị huấn luyện cho cùng căn nhà. Hơn nữa, giá trị không nằm trong dữ liệu huấn luyện cũng phù hợp với giá trị kỳ vọng.

In [ ]:
print(f"Dự đoán nhà 1000 sqft {w_final*1.0 + b_final:0.1f} Nghìn đô la")
print(f"Dự đoán nhà 1200 sqft {w_final*1.2 + b_final:0.1f} Nghìn đô la")
print(f"Dự đoán nhà 2000 sqft {w_final*2.0 + b_final:0.1f} Nghìn đô la")

<a name="toc_40291_2.6"></a>
## Vẽ đồ thị
Bạn có thể thể hiện tiến trình của gradient descent trong quá trình thực thi bằng cách vẽ cost qua các lần lặp trên một đồ thị đường đồng mức (contour plot) của cost(w,b). 

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12, 6))
plt_contour_wgrad(x_train, y_train, p_hist, ax)

Ở trên, đồ thị đường đồng mức cho thấy $cost(w,b)$ trên một phạm vi của $w$ và $b$. Các mức cost được biểu diễn bằng các vòng tròn. Được vẽ chồng lên bằng các mũi tên đỏ là đường đi của gradient descent. Dưới đây là một số điều cần lưu ý:
- Đường đi tiến triển ổn định (đơn điệu) hướng về mục tiêu.
- các bước ban đầu lớn hơn nhiều so với các bước gần mục tiêu.

**Phóng to**, chúng ta có thể thấy các bước cuối cùng của gradient descent. Lưu ý khoảng cách giữa các bước co lại khi gradient tiến gần đến 0.

In [ ]:
fig, ax = plt.subplots(1,1, figsize=(12, 4))
plt_contour_wgrad(x_train, y_train, p_hist, ax, w_range=[180, 220, 0.5], b_range=[80, 120, 0.5],
            contours=[1,5,10,20],resolution=0.5)

<a name="toc_40291_2.7.1"></a>
### Tăng Tốc độ học

<figure>
 <img align="left", src="./images/C1_W1_Lab03_alpha_too_big.PNG"   style="width:340px;height:240px;" >
</figure>
Trong bài giảng, đã có một cuộc thảo luận liên quan đến giá trị phù hợp của tốc độ học, $\alpha$ trong phương trình (3). $\alpha$ càng lớn, gradient descent sẽ hội tụ về một nghiệm càng nhanh. Nhưng, nếu nó quá lớn, gradient descent sẽ phân kỳ. Ở trên bạn có một ví dụ về một nghiệm hội tụ tốt.

Hãy thử tăng giá trị của $\alpha$ và xem điều gì xảy ra:

In [ ]:
# khởi tạo tham số
w_init = 0
b_init = 0
# đặt alpha thành một giá trị lớn
iterations = 10
tmp_alpha = 8.0e-1
# chạy gradient descent
w_final, b_final, J_hist, p_hist = gradient_descent(x_train ,y_train, w_init, b_init, tmp_alpha, 
                                                    iterations, compute_cost, compute_gradient)

Ở trên, $w$ và $b$ đang nhảy qua lại giữa dương và âm với giá trị tuyệt đối tăng lên sau mỗi lần lặp. Hơn nữa, ở mỗi lần lặp $\frac{\partial J(w,b)}{\partial w}$ đổi dấu và cost đang tăng thay vì giảm. Đây là một dấu hiệu rõ ràng cho thấy *tốc độ học quá lớn* và nghiệm đang phân kỳ. 
Hãy trực quan hóa điều này bằng một đồ thị.

In [ ]:
plt_divergence(p_hist, J_hist,x_train, y_train)
plt.show()

Ở trên, đồ thị bên trái cho thấy sự tiến triển của $w$ qua vài bước đầu tiên của gradient descent. $w$ dao động từ dương sang âm và cost tăng nhanh chóng. Gradient Descent hoạt động trên cả $w$ và $b$ đồng thời, vì vậy cần đồ thị 3D bên phải để có bức tranh đầy đủ.


## Chúc mừng!
Trong lab này bạn đã:
- đi sâu vào chi tiết của gradient descent cho một biến.
- xây dựng một hàm để tính gradient
- trực quan hóa gradient là gì
- hoàn thành một hàm gradient descent
- sử dụng gradient descent để tìm các tham số
- xem xét tác động của việc chọn kích thước tốc độ học